# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mursaleen-developer/fly-rank-ML-Intenship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

### Task type: Scoring

My lane is a **content refresh prioritization** problem, so I frame it as a **scoring task**.

The decision is: **Which content pages should an editor review first for a possible refresh?**

The model would give each content page a priority score. Pages with higher scores would be placed higher in the refresh queue.

This is a scoring problem because the goal is not only to predict whether a page is declining. The goal is to **prioritize which pages should be reviewed first**.

The output supports a content editor or SEO team in deciding which pages to investigate and potentially refresh.

In [9]:
# Basic check for our ML task framing
task_type = "scoring"
decision = "Which content pages should an editor review first for a possible refresh?"

print("Task type:", task_type)
print("Decision:", decision)


Task type: scoring
Decision: Which content pages should an editor review first for a possible refresh?


## 2. Target or proxy

### Target: `is_declining_label`

The target I would use is **`is_declining_label`**.

It is an observed outcome in the starter dataset. The label is `1` when the content item's `trend_direction` is `"down"` and `0` otherwise.

This gives us an observed signal of whether a content item is currently declining.

For the scoring task, the model can learn which characteristics are associated with declining content and use that information to assign a priority score to each page.

I will **not use `trend_direction` or `trend_pct` as model features**, because they are used to create the target and would cause data leakage.

The target is therefore based on an observed outcome rather than a rule that we invent ourselves.

In [10]:
# Load the raw starter dataset and create the observed target

import pandas as pd

data_url = "https://raw.githubusercontent.com/mursaleen-developer/fly-rank-ML-Intenship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_url)

print("Dataset shape:", df.shape)

# The raw dataset contains trend_direction.
# The repository defines the target as 1 when trend_direction == "down".
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("\nTarget column created:", "is_declining_label" in df.columns)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentages:")
print(
    df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Dataset shape: (30000, 44)

Target column created: True

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


## 3. Success metric

### Metric: Precision@K

I will use **Precision@K** as the main success metric.

The purpose of the scoring system is to prioritize pages for a content editor. Therefore, I care about how many of the highest-priority pages are actually declining.

For example, if the team reviews the top 50 pages selected by the model, Precision@50 tells us what proportion of those 50 pages are actually declining.

A higher Precision@K means the refresh queue contains more pages that are relevant to the decision.

This metric fits the task because the content team has limited time and cannot review every page. The goal is to put the most useful pages near the top of the queue.

In [11]:
# Define the success metric for our scoring task

k = 50

print("Success metric: Precision@K")
print("K =", k)
print("Meaning: Of the top", k, "pages recommended for review,")
print("how many are actually declining?")


Success metric: Precision@K
K = 50
Meaning: Of the top 50 pages recommended for review,
how many are actually declining?


## 4. The unit of analysis, as a real dataframe

### Unit of analysis: one content item/page

Each row in the starter dataset represents **one pseudonymized content item (page)**.

This is the unit at which we make the refresh-prioritization decision. Each page has its own content properties, search performance, engagement metrics, and target label.

The dataset contains 30,000 content items.

For this task, the model would assign a priority score to each content item so that the content team can rank pages and decide which ones to review first.

The identifiers such as `content_id` and `client_id` are used for identification and grouping, not as predictive features.

In [12]:
# Show the unit of analysis as a real dataframe

print("Dataset shape:", df.shape)

print("\nOne row represents: ONE CONTENT ITEM / PAGE")

# Show a small sample of the actual dataframe
display(
    df[
        [
            "content_id",
            "client_id",
            "content_type",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "trend_direction",
            "is_declining_label"
        ]
    ].head(10)
)
# Show the unit of analysis as a real dataframe

print("Dataset shape:", df.shape)

print("\nOne row represents: ONE CONTENT ITEM / PAGE")

display(
    df[
        [
            "content_id",
            "client_id",
            "content_type",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "trend_direction",
            "is_declining_label"
        ]
    ].head(10)
)

# Check whether each content_id appears once
duplicate_content_ids = df["content_id"].duplicated().sum()

print("\nDuplicate content IDs:", duplicate_content_ids)

Dataset shape: (30000, 45)

One row represents: ONE CONTENT ITEM / PAGE


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,1,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,3,down,1


Dataset shape: (30000, 45)

One row represents: ONE CONTENT ITEM / PAGE


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,1,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,3,down,1



Duplicate content IDs: 0


## 5. Why ML beats a fixed rule here

A simple fixed rule could be:

> "If a page's impressions decrease by more than 20%, put it in the refresh queue."

This rule is easy to understand, but it relies mainly on one threshold.

Content performance can be influenced by several signals at the same time, including impressions, clicks, sessions, CTR, average position, content age, days since the last update, search volume, competition, and content type.

A scoring model can combine these signals and learn patterns associated with declining content instead of relying on one manually chosen cutoff.

The ML output would support a content editor or SEO team by ranking pages for review. It would not automatically decide that a page must be refreshed.

ML is useful here only if it improves the prioritization decision compared with a simple baseline rule. The model should therefore be evaluated using Precision@K on the pages near the top of the review queue.

A wrong prioritization can waste editor time by sending healthy pages to the top of the queue, while a missed declining page can delay a useful refresh. Therefore, the goal is decision support rather than automatic content changes.

In [13]:
# Compare a simple fixed-rule idea with the ML scoring idea

rule_description = "Refresh priority if impressions decline by more than 20%"

ml_description = (
    "Combine multiple content and performance signals "
    "to produce a priority score"
)

print("Fixed-rule baseline:")
print(rule_description)

print("\nML scoring approach:")
print(ml_description)

print("\nEvaluation metric: Precision@50")
print("Decision supported: Which pages should be reviewed first?")


Fixed-rule baseline:
Refresh priority if impressions decline by more than 20%

ML scoring approach:
Combine multiple content and performance signals to produce a priority score

Evaluation metric: Precision@50
Decision supported: Which pages should be reviewed first?


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.